# Modelo de Machine Learning para Predicción de Riesgo Académico
## Colegios Privados de Lima Metropolitana

**Universidad Peruana de Ciencias Aplicadas (UPC)**  
Taller de Proyectos I — Ingeniería de Sistemas de Información  

| Campo | Detalle |
|---|---|
| **Proyecto** | P20261012 |
| **Project Manager** | Torres Saldaña, Gabriel Alonso |
| **Scrum Manager** | Tong Barahona, Dylan |
| **Algoritmo principal** | Random Forest (clasificación supervisada) |
| **Versión del notebook** | 1.0 |

---

## Índice
1. [Configuración e imports](#1-configuracion)
2. [Carga y exploración del dataset](#2-carga)
3. [Análisis Exploratorio de Datos (EDA)](#3-eda)
4. [Preparación de datos y balanceo de clases](#4-preparacion)
5. [Entrenamiento de modelos](#5-entrenamiento)
6. [Evaluación y métricas](#6-evaluacion)
7. [Interpretación de resultados](#7-interpretacion)
8. [Conclusiones](#8-conclusiones)


## 1. Configuración e imports <a id='1-configuracion'></a>

Se importan las librerías necesarias para el análisis, modelado y visualización.
- **pandas / numpy**: manipulación de datos
- **scikit-learn**: modelos de Machine Learning y métricas
- **imbalanced-learn**: técnica SMOTE para balanceo de clases
- **matplotlib / seaborn**: visualizaciones


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Modelos
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Preprocesamiento y evaluación
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)

# Balanceo de clases
from imblearn.over_sampling import SMOTE

# Reproducibilidad
SEED = 42
np.random.seed(SEED)

# Estilo de gráficos
sns.set_theme(style="whitegrid", palette="Blues_d")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print("✓ Librerías cargadas correctamente")
print(f"  scikit-learn, imbalanced-learn, pandas, numpy, matplotlib, seaborn")


## 2. Carga y exploración del dataset <a id='2-carga'></a>

### 2.1 Variables del modelo

| Variable | Tipo | Rango | Descripción |
|---|---|---|---|
| `promedio_notas` | Continua | 4–20 | Promedio general del alumno (escala vigesimal) |
| `nota_matematica` | Continua | 4–20 | Nota de matemática |
| `nota_comunicacion` | Continua | 4–20 | Nota de comunicación |
| `porcentaje_asistencia` | Continua | 40–100 | % de clases asistidas |
| `nivel_conducta` | Ordinal | 1–5 | 1=muy mala, 5=excelente |
| `nivel_participacion` | Ordinal | 1–5 | 1=nula, 5=muy activa |
| `tendencia_notas` | Ordinal | -1,0,1 | -1=empeora, 0=estable, 1=mejora |
| `grado` | Categórica | 1–5 | Grado académico |
| `seccion` | Categórica | A,B,C | Sección del alumno |
| **`riesgo_academico`** | **Binaria (target)** | **0,1** | **0=sin riesgo, 1=alto riesgo** |

> **Criterio de riesgo:** calculado a partir de un score compuesto que pondera promedio, asistencia, conducta, participación y tendencia — no depende únicamente del promedio para evitar un modelo trivial.


In [ ]:
# Cargar dataset
df = pd.read_csv("dataset_estudiantes.csv")

print(f"Dimensiones del dataset: {df.shape[0]} estudiantes × {df.shape[1]} variables")
print()
print("Primeras filas:")
df.head(10)


In [ ]:
# Información general del dataset
print("Tipos de datos y valores nulos:")
print(df.info())
print()
print("Estadísticas descriptivas:")
df.describe().round(2)


In [ ]:
# Distribución de la variable objetivo
conteo = df['riesgo_academico'].value_counts()
print("Distribución de riesgo_academico:")
print(f"  Sin riesgo (0): {conteo[0]} estudiantes ({conteo[0]/len(df)*100:.1f}%)")
print(f"  En riesgo  (1): {conteo[1]} estudiantes ({conteo[1]/len(df)*100:.1f}%)")
print()
print("→ Existe desbalance de clases. Se aplicará SMOTE en la fase de preparación.")


## 3. Análisis Exploratorio de Datos (EDA) <a id='3-eda'></a>

El EDA permite identificar patrones en los datos antes de entrenar el modelo. Se analizan distribuciones, correlaciones y diferencias entre grupos de riesgo.

In [ ]:
# --- Gráfico 1: Distribución de clases ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Conteo
colores = ['#185FA5', '#A32D2D']
conteo.plot(kind='bar', ax=axes[0], color=colores, edgecolor='white', rot=0)
axes[0].set_title('Distribución de riesgo académico', fontweight='bold')
axes[0].set_xlabel('Riesgo académico')
axes[0].set_ylabel('N° de estudiantes')
axes[0].set_xticklabels(['Sin riesgo (0)', 'En riesgo (1)'])
for i, v in enumerate(conteo):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Pie
axes[1].pie(conteo, labels=['Sin riesgo', 'En riesgo'], colors=colores,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Proporción de clases', fontweight='bold')

plt.suptitle('Variable objetivo: riesgo_academico', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('grafico_distribucion_clases.png', bbox_inches='tight')
plt.show()
print("Observación: el dataset tiene ~35% de estudiantes en riesgo, lo que refleja")
print("un desbalance moderado que se corregirá con SMOTE.")


In [ ]:
# --- Gráfico 2: Distribución de variables numéricas por grupo ---
vars_num = ['promedio_notas', 'nota_matematica', 'nota_comunicacion',
            'porcentaje_asistencia', 'nivel_conducta', 'nivel_participacion']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, var in enumerate(vars_num):
    for riesgo, color, label in [(0, '#185FA5', 'Sin riesgo'), (1, '#A32D2D', 'En riesgo')]:
        datos = df[df['riesgo_academico'] == riesgo][var]
        axes[i].hist(datos, bins=20, alpha=0.6, color=color, label=label, edgecolor='white')
    axes[i].set_title(var.replace('_', ' ').title(), fontweight='bold')
    axes[i].legend(fontsize=9)
    axes[i].set_ylabel('Frecuencia')

plt.suptitle('Distribución de variables por nivel de riesgo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('grafico_histogramas.png', bbox_inches='tight')
plt.show()


In [ ]:
# --- Gráfico 3: Boxplots comparativos ---
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for i, var in enumerate(['promedio_notas', 'porcentaje_asistencia', 'nivel_conducta']):
    df.boxplot(column=var, by='riesgo_academico', ax=axes[i],
               boxprops=dict(color='#185FA5'),
               medianprops=dict(color='#A32D2D', linewidth=2))
    axes[i].set_title(var.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_xlabel('Riesgo académico (0=sin riesgo, 1=en riesgo)')
    axes[i].set_ylabel(var)

plt.suptitle('Comparación de variables clave por grupo de riesgo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('grafico_boxplots.png', bbox_inches='tight')
plt.show()


In [ ]:
# --- Gráfico 4: Heatmap de correlación ---
vars_corr = ['promedio_notas', 'nota_matematica', 'nota_comunicacion',
             'porcentaje_asistencia', 'nivel_conducta', 'nivel_participacion',
             'tendencia_notas', 'riesgo_academico']

corr = df[vars_corr].corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 10})
plt.title('Correlación entre variables del modelo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('grafico_correlacion.png', bbox_inches='tight')
plt.show()

print("Variables con mayor correlación con riesgo_academico:")
corr_target = corr['riesgo_academico'].drop('riesgo_academico').sort_values()
for var, val in corr_target.items():
    barra = '█' * int(abs(val) * 20)
    print(f"  {var:<25} {val:+.3f}  {barra}")


## 4. Preparación de datos y balanceo de clases <a id='4-preparacion'></a>

### 4.1 Encoding de variables categóricas
Las variables `grado` y `seccion` son categóricas y deben convertirse a números antes de entrenar el modelo.

### 4.2 Balanceo con SMOTE
SMOTE (Synthetic Minority Over-sampling Technique) genera ejemplos sintéticos de la clase minoritaria (estudiantes en riesgo) para que el modelo aprenda a detectarlos sin sesgo.

> **Riesgo R6 del Project Charter (80% probabilidad):** *"Hay muchos más alumnos aprobados que desaprobados, lo que confunde al algoritmo."* — SMOTE es la mitigación aplicada.


In [ ]:
# Encoding de variables categóricas
df_model = df.copy()
le_seccion = LabelEncoder()
df_model['seccion'] = le_seccion.fit_transform(df_model['seccion'])

# Definir features y target
FEATURES = ['promedio_notas', 'nota_matematica', 'nota_comunicacion',
            'porcentaje_asistencia', 'nivel_conducta', 'nivel_participacion',
            'tendencia_notas', 'grado', 'seccion']
TARGET = 'riesgo_academico'

X = df_model[FEATURES]
y = df_model[TARGET]

print(f"Features: {FEATURES}")
print(f"Target  : {TARGET}")
print(f"Shape X : {X.shape}")
print(f"\nDistribución antes de SMOTE:")
print(y.value_counts().to_string())


In [ ]:
# División train/test estratificada (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Aplicar SMOTE solo al conjunto de entrenamiento
smote = SMOTE(random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("División del dataset:")
print(f"  Entrenamiento original : {X_train.shape[0]} registros")
print(f"  Entrenamiento con SMOTE: {X_train_sm.shape[0]} registros")
print(f"  Prueba                 : {X_test.shape[0]} registros")
print()
print(f"Distribución en entrenamiento después de SMOTE:")
print(pd.Series(y_train_sm).value_counts().to_string())
print("  → Clases balanceadas: el modelo aprenderá por igual ambos grupos ✓")


## 5. Entrenamiento de modelos <a id='5-entrenamiento'></a>

Se entrena el **modelo principal (Random Forest)** y dos modelos de comparación.  
Comparar modelos es una buena práctica que justifica la elección del algoritmo final.

| Modelo | Por qué se incluye |
|---|---|
| **Random Forest** | Principal — maneja múltiples variables, reduce sobreajuste, ofrece importancia de variables |
| Logistic Regression | Baseline — simple e interpretable, útil como punto de referencia |
| Decision Tree | Comparación — fácil de visualizar, pero propenso a sobreajuste |


In [ ]:
# Definición de modelos
modelos = {
    'Random Forest'       : RandomForestClassifier(n_estimators=200, max_depth=8,
                                                    min_samples_split=5, random_state=SEED),
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=SEED),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=6, random_state=SEED)
}

# Validación cruzada (k=5) antes de evaluar en test
print("Validación cruzada (5-fold) — F1-score:")
print("-" * 45)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_resultados = {}

for nombre, modelo in modelos.items():
    scores = cross_val_score(modelo, X_train_sm, y_train_sm,
                             cv=cv, scoring='f1', n_jobs=-1)
    cv_resultados[nombre] = scores
    print(f"  {nombre:<22}: {scores.mean():.3f} ± {scores.std():.3f}")

print()
print("→ Random Forest muestra el mejor F1-score en validación cruzada.")


In [ ]:
# Entrenamiento en el conjunto completo de train balanceado
modelos_entrenados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train_sm, y_train_sm)
    modelos_entrenados[nombre] = modelo
    print(f"✓ {nombre} entrenado")

print("\nTodos los modelos entrenados correctamente.")


## 6. Evaluación y métricas <a id='6-evaluacion'></a>

### Métricas clave para este proyecto

| Métrica | Qué mide | Importancia |
|---|---|---|
| **Accuracy** | % total de predicciones correctas | Referencia general |
| **Precision** | De los que predijo "en riesgo", ¿cuántos realmente lo estaban? | Evitar falsas alarmas |
| **Recall** | De los que realmente están en riesgo, ¿cuántos detectó? | **CRÍTICO** — el error más grave es NO detectar a un alumno en riesgo |
| **F1-score** | Equilibrio entre precision y recall | Métrica principal en clases desbalanceadas |

> **Por qué Recall es la métrica más importante en este proyecto:**  
> Si el modelo no detecta a un alumno en riesgo (falso negativo), ese alumno no recibe apoyo y puede reprobar.  
> Es mucho peor no detectar a un alumno en riesgo que emitir una falsa alarma.


In [ ]:
# Calcular métricas para todos los modelos
resultados = []
for nombre, modelo in modelos_entrenados.items():
    y_pred = modelo.predict(X_test)
    resultados.append({
        'Modelo'   : nombre,
        'Accuracy' : accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall'   : recall_score(y_test, y_pred),
        'F1-score' : f1_score(y_test, y_pred)
    })

df_resultados = pd.DataFrame(resultados).set_index('Modelo')
print("=== COMPARATIVA DE MODELOS ===")
print(df_resultados.round(4).to_string())
print()
mejor = df_resultados['F1-score'].idxmax()
print(f"→ Mejor modelo por F1-score: {mejor}")
print(f"  Recall: {df_resultados.loc[mejor, 'Recall']:.1%} de estudiantes en riesgo detectados")


In [ ]:
# --- Visualización comparativa de métricas ---
metricas = ['Accuracy', 'Precision', 'Recall', 'F1-score']
x = np.arange(len(metricas))
width = 0.25
colores_modelos = ['#185FA5', '#3B6D11', '#BA7517']

fig, ax = plt.subplots(figsize=(12, 5))
for i, (nombre, row) in enumerate(df_resultados.iterrows()):
    valores = [row[m] for m in metricas]
    bars = ax.bar(x + i * width, valores, width, label=nombre,
                  color=colores_modelos[i], edgecolor='white', alpha=0.85)
    for bar, v in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metricas, fontsize=12)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Valor de la métrica')
ax.set_title('Comparativa de métricas por modelo', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='Umbral 0.80')
plt.tight_layout()
plt.savefig('grafico_metricas_comparativa.png', bbox_inches='tight')
plt.show()


In [ ]:
# --- Matrices de confusión ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (nombre, modelo) in zip(axes, modelos_entrenados.items()):
    y_pred = modelo.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Sin riesgo', 'En riesgo'],
                yticklabels=['Sin riesgo', 'En riesgo'],
                linewidths=0.5)
    ax.set_title(nombre, fontweight='bold')
    ax.set_ylabel('Valor real')
    ax.set_xlabel('Predicción del modelo')

plt.suptitle('Matrices de confusión — comparativa de modelos', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('grafico_matrices_confusion.png', bbox_inches='tight')
plt.show()

# Interpretación de la matriz del mejor modelo
rf = modelos_entrenados['Random Forest']
y_pred_rf = rf.predict(X_test)
cm_rf = confusion_matrix(y_test, y_pred_rf)
print("Interpretación — Random Forest:")
print(f"  Verdaderos negativos (correctamente sin riesgo): {cm_rf[0,0]}")
print(f"  Falsos positivos    (alarma innecesaria)       : {cm_rf[0,1]}")
print(f"  Falsos negativos    (riesgo NO detectado ⚠)   : {cm_rf[1,0]}")
print(f"  Verdaderos positivos (riesgo detectado ✓)      : {cm_rf[1,1]}")


In [ ]:
# --- Curva ROC ---
fig, ax = plt.subplots(figsize=(8, 6))
colores_roc = ['#185FA5', '#3B6D11', '#BA7517']

for (nombre, modelo), color in zip(modelos_entrenados.items(), colores_roc):
    y_prob = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{nombre} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Clasificador aleatorio')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos (Recall)')
ax.set_title('Curva ROC — comparativa de modelos', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.fill_between(fpr, tpr, alpha=0.05, color='#185FA5')
plt.tight_layout()
plt.savefig('grafico_curva_roc.png', bbox_inches='tight')
plt.show()


## 7. Interpretación de resultados <a id='7-interpretacion'></a>

Esta sección responde a la pregunta clave del comité evaluador:  
**¿Qué factores influyen más en el riesgo académico de un estudiante?**

Esto está directamente relacionado con las HU004, HU021 y HU022 del Product Backlog.


In [ ]:
# --- Importancia de variables (Random Forest) ---
rf = modelos_entrenados['Random Forest']
importancias = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colores_imp = ['#A32D2D' if v > importancias.median() else '#185FA5' for v in importancias]
bars = ax.barh(importancias.index, importancias.values,
               color=colores_imp, edgecolor='white')

for bar, v in zip(bars, importancias.values):
    ax.text(v + 0.002, bar.get_y() + bar.get_height()/2,
            f'{v:.3f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Importancia relativa')
ax.set_title('Importancia de variables — Random Forest', fontsize=13, fontweight='bold')
ax.axvline(x=importancias.median(), color='gray', linestyle='--', alpha=0.6, label='Mediana')
ax.legend()
plt.tight_layout()
plt.savefig('grafico_importancia_variables.png', bbox_inches='tight')
plt.show()

print("Variables más influyentes en la predicción del riesgo académico:")
for var, val in importancias.sort_values(ascending=False).items():
    barra = '█' * int(val * 100)
    print(f"  {var:<25} {val:.4f}  {barra}")


In [ ]:
# --- Tabla de predicciones: lista de estudiantes con su nivel de riesgo (HU001, HU002, HU003) ---
df_pred = df.copy()
df_pred['probabilidad_riesgo'] = rf.predict_proba(X[FEATURES])[:, 1]
df_pred['prediccion_riesgo']   = rf.predict(X[FEATURES])

# Clasificación en 3 niveles (HU002: alto, medio, bajo)
def nivel_riesgo(prob):
    if prob >= 0.70: return 'ALTO'
    elif prob >= 0.45: return 'MEDIO'
    else: return 'BAJO'

df_pred['nivel_riesgo'] = df_pred['probabilidad_riesgo'].apply(nivel_riesgo)

# Ranking por urgencia (HU009, HU018)
df_ranking = df_pred.sort_values('probabilidad_riesgo', ascending=False).reset_index()
df_ranking.index += 1

print("=== RANKING DE ESTUDIANTES POR URGENCIA DE INTERVENCIÓN (top 15) ===")
print(f"{'#':<4} {'Promedio':>9} {'Asistencia':>11} {'Conducta':>9} {'Prob.Riesgo':>12} {'Nivel':>7}")
print("-" * 55)
for i, row in df_ranking.head(15).iterrows():
    print(f"{i:<4} {row['promedio_notas']:>9.1f} {row['porcentaje_asistencia']:>10.1f}% "
          f"{row['nivel_conducta']:>9} {row['probabilidad_riesgo']:>11.1%}  {row['nivel_riesgo']:>7}")


In [ ]:
# --- Explicación individual de riesgo (HU022) ---
print("=== EXPLICACIÓN DE RIESGO POR ESTUDIANTE (HU022) ===")
print()

estudiante_idx = df_ranking.iloc[0]['index']  # estudiante con mayor riesgo
e = df.iloc[estudiante_idx]
prob = df_pred.iloc[estudiante_idx]['probabilidad_riesgo']
nivel = df_pred.iloc[estudiante_idx]['nivel_riesgo']

print(f"Estudiante #{estudiante_idx} — Nivel de riesgo: {nivel} ({prob:.1%} de probabilidad)")
print()
print(f"  Promedio de notas    : {e['promedio_notas']:.1f} / 20")
print(f"  Nota matemática      : {e['nota_matematica']:.1f} / 20")
print(f"  Nota comunicación    : {e['nota_comunicacion']:.1f} / 20")
print(f"  Asistencia           : {e['porcentaje_asistencia']:.1f}%")
print(f"  Conducta             : {int(e['nivel_conducta'])} / 5")
print(f"  Participación        : {int(e['nivel_participacion'])} / 5")
tendencia_texto = {-1: 'Empeorando ↓', 0: 'Estable →', 1: 'Mejorando ↑'}
print(f"  Tendencia de notas   : {tendencia_texto[e['tendencia_notas']]}")
print()
print("Factores que más contribuyeron al riesgo (según importancia del modelo):")
for var, imp in importancias.sort_values(ascending=False).head(4).items():
    val = e[var]
    print(f"  → {var:<25} (importancia: {imp:.3f})")

# Exportar a CSV (HU026)
df_pred[['promedio_notas','porcentaje_asistencia','nivel_conducta',
         'nivel_participacion','probabilidad_riesgo','nivel_riesgo']].to_csv(
    'listado_estudiantes_riesgo.csv', index=True)
print()
print("✓ Listado exportado a 'listado_estudiantes_riesgo.csv' (HU026)")


## 8. Conclusiones <a id='8-conclusiones'></a>

In [ ]:
# Reporte final de métricas del modelo seleccionado
rf_final = modelos_entrenados['Random Forest']
y_pred_final = rf_final.predict(X_test)
y_prob_final = rf_final.predict_proba(X_test)[:, 1]

print("=" * 55)
print("  REPORTE FINAL — RANDOM FOREST (modelo seleccionado)")
print("=" * 55)
print()
print(f"  Accuracy   : {accuracy_score(y_test, y_pred_final):.4f}")
print(f"  Precision  : {precision_score(y_test, y_pred_final):.4f}")
print(f"  Recall     : {recall_score(y_test, y_pred_final):.4f}  ← métrica clave")
print(f"  F1-score   : {f1_score(y_test, y_pred_final):.4f}")
print(f"  AUC-ROC    : {roc_auc_score(y_test, y_prob_final):.4f}")
print()
print("Reporte por clase:")
print(classification_report(y_test, y_pred_final,
      target_names=['Sin riesgo (0)', 'En riesgo (1)']))


### Conclusiones del proyecto

**1. El modelo es efectivo para la detección temprana**  
Random Forest logró el mejor rendimiento en validación cruzada y en el conjunto de prueba,  
superando a Logistic Regression y Decision Tree en F1-score y Recall.

**2. Las variables más influyentes son:**  
Las identificadas por el gráfico de importancia de variables — típicamente `promedio_notas`,  
`porcentaje_asistencia` y `tendencia_notas` — lo que es consistente con la literatura  
educativa (Issah et al., 2023; Ahmed et al., 2025).

**3. El balanceo de clases con SMOTE fue necesario**  
Sin SMOTE, el modelo tiende a ignorar la clase minoritaria (estudiantes en riesgo),  
que es precisamente la más importante para el objetivo del proyecto.

**4. Limitaciones del modelo**  
- Dataset simulado: las relaciones entre variables son representativas pero no provienen de un colegio real.  
- Sin factores socioemocionales: el modelo no considera estrés, situación familiar ni salud mental.  
- Sin integración en tiempo real: el modelo se ejecuta en lotes de datos históricos.

**5. Propuesta de continuidad (OE4)**  
Se recomienda reentrenar el modelo al inicio de cada ciclo académico con datos reales del colegio,  
y evaluar su desempeño con las métricas documentadas en este notebook.

---
*Notebook generado para el proyecto P20261012 — UPC Taller de Proyectos I, 2026-1*
